In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from datetime import date, timedelta

In [ ]:
"""
VERSION INFORMATION
"""

%load_ext watermark

%watermark --python --packages numpy,pandas,xarray,matplotlib,seaborn,scipy,sklearn,tensorflow,pytorch,jupyterlab --machine --iversions

In [ ]:
with open("bank-holidays.json", "r") as f:
    data = json.load(f)

# Access a specific division's events as a DataFrame
# Only interested in Eng, Wales, Scot (GB)
df_eng_wales = pd.DataFrame(data["england-and-wales"]["events"])
df_scot = pd.DataFrame(data["scotland"]["events"])

In [ ]:
dfs = {
    "df_eng_wales" : df_eng_wales,
    "df_scot" : df_scot
}

In [ ]:
for name, df in dfs.items():
    print(f"=== {name} ===")
    print(df.head())
    print("---")
    print(df.info())
    print("---")
    print(f"Date range: {min(df['date'])} - {max(df['date'])}")
    print("---")
    print(f"Missing data: {df.isna().sum()}")
    print("---")

In [ ]:
"""
EASTER ALGORITHM
"""

def easter_sunday(year: int) -> date:
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    return date(year, month, day)

In [ ]:
"""
CREATE FUNCTIONS FOR USEFUL DATES
"""

def first_monday_of_month(year, month):
    d = date(year, month, 1)
    return d + timedelta(days=(7 - d.weekday()) % 7)

def last_monday_of_month(year, month):
    d = date(year + 1, 1, 1) - timedelta(days=1) if month == 12 \
        else date(year, month + 1, 1) - timedelta(days=1)
    return d - timedelta(days=d.weekday() % 7)

def sub(d):
    """Weekend substitute: Sat -> Mon, Sun -> Mon."""
    if d.weekday() == 5: return d + timedelta(days=2)
    if d.weekday() == 6: return d + timedelta(days=1)
    return d

def christmas_boxing_substitutes(year):
    xmas  = date(year, 12, 25)
    wd = xmas.weekday()
    if wd == 4: # Fri: Boxing (Sat) -> Mon
        return xmas, date(year, 12, 28)
    elif wd == 5: # Sat: Christmas -> Mon, Boxing -> Tue
        return date(year, 12, 27), date(year, 12, 28)
    elif wd == 6: # Sun: Christmas -> Tue, Boxing stays Mon
        return date(year, 12, 27), date(year, 12, 26)
    else:
        return xmas, date(year, 12, 26)

def new_year_ew(year):
    return sub(date(year, 1, 1))

def new_year_scotland(year):
    ny = date(year, 1, 1)
    wd = ny.weekday()
    if wd == 5: # Sat: NY -> Mon Jan3, Jan2(Sun) -> Tue Jan4
        return date(year, 1, 3), date(year, 1, 4)
    elif wd == 6: # Sun: Mon is 2nd Jan observed, NY sub = Tue Jan3
        return date(year, 1, 3), date(year, 1, 2)
    elif wd == 4: # Fri: NY normal, Jan2(Sat) -> Mon Jan4
        return ny, date(year, 1, 4)
    else:
        return ny, date(year, 1, 2)

In [ ]:
"""
ONE-OFF SPECIAL HOLS
"""

SPECIAL_EW = {
    date(2011, 4, 29), # Royal Wedding
    date(2012, 6, 5), # Diamond Jubilee extra
    date(2022, 6, 3), # Platinum Jubilee extra
    date(2022, 9, 19), # State Funeral of Queen Elizabeth II
    date(2023, 5, 8), # Coronation of King Charles III
}
SPECIAL_SCOTLAND = {
    date(2011, 4, 29),
    date(2012, 6, 5),
    date(2022, 6, 3),
    date(2022, 9, 19),
    date(2023, 5, 8),
}

# Years where the Spring or Early May BH moved
SPRING_BH_OVERRIDES_EW = {2012: date(2012, 6, 4), 2022: date(2022, 6, 2)}
SPRING_BH_OVERRIDES_SCOTLAND = {2012: date(2012, 6, 4), 2022: date(2022, 6, 2)}
EARLY_MAY_OVERRIDES_EW = {2020: date(2020, 5, 8)} # VE Day 75
EARLY_MAY_OVERRIDES_SCOTLAND = {2020: date(2020, 5, 8)}

In [ ]:
"""
HOLIDAY GENERATORS
"""

def england_wales_holidays(year):
    e = easter_sunday(year)
    h = set()
    h.add(new_year_ew(year))
    h.add(e - timedelta(days=2)) # Good Friday
    h.add(e + timedelta(days=1)) # Easter Monday
    h.add(EARLY_MAY_OVERRIDES_EW.get(year, first_monday_of_month(year, 5)))
    h.add(SPRING_BH_OVERRIDES_EW.get(year, last_monday_of_month(year, 5)))
    h.add(last_monday_of_month(year, 8)) # Summer BH
    xmas, boxing = christmas_boxing_substitutes(year)
    h.add(xmas); h.add(boxing)
    h |= {d for d in SPECIAL_EW if d.year == year}
    return h

def scotland_holidays(year):
    e = easter_sunday(year)
    h = set()
    ny, jan2 = new_year_scotland(year)
    h.add(ny); h.add(jan2)
    h.add(e - timedelta(days=2)) # Good Friday (no Easter Monday)
    h.add(EARLY_MAY_OVERRIDES_SCOTLAND.get(year, first_monday_of_month(year, 5)))
    h.add(SPRING_BH_OVERRIDES_SCOTLAND.get(year, last_monday_of_month(year, 5)))
    h.add(first_monday_of_month(year, 8)) # Summer BH (first Mon Aug in Scotland)
    h.add(sub(date(year, 11, 30))) # St Andrew's Day
    xmas, boxing = christmas_boxing_substitutes(year)
    h.add(xmas); h.add(boxing)
    h |= {d for d in SPECIAL_SCOTLAND if d.year == year}
    return h

In [ ]:
"""
VERFIY ALGORITHM
"""

with open("bank-holidays.json", "r") as f:
    raw = json.load(f)

json_ew = {date.fromisoformat(e["date"]) for e in raw["england-and-wales"]["events"]}
json_sc = {date.fromisoformat(e["date"]) for e in raw["scotland"]["events"]}
# Northern Ireland not loaded at all

JSON_START, JSON_END = date(2019, 1, 1), date(2028, 12, 31)
print(f"JSON covers {JSON_START} - {JSON_END}")

# Compare algorithm vs JSON for overlapping years
algo_ew = {d for yr in range(2019, 2029) for d in england_wales_holidays(yr)}
algo_sc = {d for yr in range(2019, 2029) for d in scotland_holidays(yr)}
algo_ew = {d for d in algo_ew if JSON_START <= d <= JSON_END}
algo_sc = {d for d in algo_sc if JSON_START <= d <= JSON_END}

# Should see 4 empty lists
print("\nE&W  — in JSON but not algorithm:", sorted(json_ew - algo_ew))
print("E&W  — in algorithm but not JSON:", sorted(algo_ew - json_ew))
print("Scot — in JSON but not algorithm:", sorted(json_sc - algo_sc))
print("Scot — in algorithm but not JSON:", sorted(algo_sc - json_sc))

In [ ]:
"""
BUILD CALENDAR FOR FULL PERIOD 2010 - 2045
"""

all_dates = pd.date_range("2010-01-01", "2045-12-31", freq="D")
df = pd.DataFrame({"date": all_dates.date})

# Pre-compute algorithmic sets for full range
algo_ew_all = {d for yr in range(2010, 2046) for d in england_wales_holidays(yr)}
algo_sc_all = {d for yr in range(2010, 2046) for d in scotland_holidays(yr)}

def is_hol_ew(d):
    return int(d in json_ew if JSON_START <= d <= JSON_END else d in algo_ew_all)

def is_hol_sc(d):
    return int(d in json_sc if JSON_START <= d <= JSON_END else d in algo_sc_all)

df["is_holiday_eng_wales"] = df["date"].apply(is_hol_ew)
df["is_holiday_scotland"]  = df["date"].apply(is_hol_sc)
df["is_holiday_gb_any"]    = ((df["is_holiday_eng_wales"] == 1) |
                               (df["is_holiday_scotland"]  == 1)).astype(int)

df["date"] = pd.to_datetime(df["date"])
print(df.head(10))

In [ ]:
"""
CHECKS
"""

expected = pd.date_range("2010-01-01", "2045-12-31", freq="D")

print(f"Rows: {len(df)} (expected {len(expected)}) {'AS EXPECTED' if len(df)==len(expected) else 'UNEXPECTED'}")
print(f"Duplicate dates: {df['date'].duplicated().sum()} {'AS EXPECTED' if df['date'].duplicated().sum()==0 else 'UNEXPECTED'}")
print(f"NaN values: {df.isna().sum().sum()} {'AS EXPECTED' if df.isna().sum().sum()==0 else 'UNEXPECTED'}")
print(f"E&W holidays: {df['is_holiday_eng_wales'].sum()}")
print(f"Scotland: {df['is_holiday_scotland'].sum()}")
print(f"GB any: {df['is_holiday_gb_any'].sum()}")

In [ ]:
df.to_csv("holiday_calendar_2010_2045.csv", index=False, date_format="%Y-%m-%d")
print("Saved: holiday_calendar_2010_2045.csv")

# Verify reload
check = pd.read_csv("holiday_calendar_2010_2045.csv")
print(check.shape) # Expect (13149, 4)
print(check.dtypes)

In [ ]:
# Collect stats from the df in memory
total_rows = len(df)
dupes = df["date"].duplicated().sum()
nan_count = df.isna().sum().sum()
ew_total = int(df["is_holiday_eng_wales"].sum())
sc_total = int(df["is_holiday_scotland"].sum())
gb_any_total = int(df["is_holiday_gb_any"].sum())

readme = f"""# GB Bank Holiday Calendar 2010–2045

## File
holiday_calendar_2010_2045.csv

## Columns
| Column                | Type    | Description |
|-----------------------|---------|-------------|
| date                  | date    | One row per calendar day, 2010-01-01 to 2045-12-31 |
| is_holiday_eng_wales  | int 0/1 | 1 if a bank holiday in England & Wales |
| is_holiday_scotland   | int 0/1 | 1 if a bank holiday in Scotland |
| is_holiday_gb_any     | int 0/1 | 1 if a bank holiday in either England & Wales or Scotland |

Northern Ireland is excluded entirely — not loaded, not used.

---

## Sources and methodology

### 2019–2028 — trusted source
GOV.UK bank-holidays.json (https://www.gov.uk/bank-holidays.json).
England & Wales and Scotland divisions used directly as the authoritative source.
Northern Ireland division was discarded at load time.

### 2010–2018 and 2029–2045 — algorithmic
Bank holidays computed using standard UK rules.

**England & Wales rules**
- New Year's Day (1 Jan, with weekend substitute)
- Good Friday (Easter − 2 days)
- Easter Monday (Easter + 1 day)
- Early May bank holiday (first Monday in May)
- Spring bank holiday (last Monday in May)
- Summer bank holiday (last Monday in August)
- Christmas Day (25 Dec, with weekend substitute)
- Boxing Day (26 Dec, with weekend substitute)

**Scotland differences from E&W**
- 2nd January added (with weekend substitute)
- Easter Monday not observed
- Summer bank holiday = first Monday in August (not last)
- St Andrew's Day added (30 Nov, with weekend substitute)

Easter dates use the Meeus/Jones/Butcher anonymous Gregorian algorithm.

---

## Known one-off special holidays hardcoded (2010–2028)
The following non-standard bank holidays cannot be derived algorithmically
and are hardcoded for the period outside JSON coverage:

| Date       | Description                                | Nations        |
|------------|--------------------------------------------|----------------|
| 2011-04-29 | Royal Wedding (Prince William & Catherine) | E&W, Scotland  |
| 2012-06-04 | Diamond Jubilee — Spring BH moved to Jun 4 | E&W, Scotland  |
| 2012-06-05 | Diamond Jubilee — extra bank holiday       | E&W, Scotland  |
| 2020-05-08 | VE Day 75 — Early May BH moved from May 4  | E&W, Scotland  |
| 2022-06-02 | Platinum Jubilee — Spring BH moved to Jun 2| E&W, Scotland  |
| 2022-06-03 | Platinum Jubilee — extra bank holiday      | E&W, Scotland  |
| 2022-09-19 | State Funeral of Queen Elizabeth II        | E&W, Scotland  |
| 2023-05-08 | Coronation of King Charles III             | E&W, Scotland  |

---

## Limitation for 2029–2045
Future special or one-off bank holidays (royal events, jubilees, national
occasions) cannot be known in advance and are NOT included for 2029–2045.
This calendar should be updated manually when such dates are announced.

---

## Verification
The algorithm was validated against the GOV.UK JSON for the overlapping
years 2019–2028. All dates matched exactly — zero discrepancies for both
England & Wales and Scotland.

---

## Quality checks
| Check                  | Result                        |
|------------------------|-------------------------------|
| Total rows             | {total_rows:,} (expected 13,149) |
| Duplicate dates        | {dupes}                            |
| Missing values         | {nan_count}                            |
| E&W holidays           | {ew_total}                           |
| Scotland holidays      | {sc_total}                           |
| GB any holidays        | {gb_any_total}                           |
"""

with open("holiday_calendar_README.md", "w") as f:
    f.write(readme)

print(readme)
print("Saved: holiday_calendar_README.md")